# LOSO (Kaggle or Colab GPU)

`python train.py --loso`: one classifier train, then leave-one-session-out on train+val. Test files stay sealed.

AuraNet is small. A free T4 is enough. CPU LOSO is hours; do not run it on a laptop as the default.

1. Runtime / accelerator: GPU.
2. Do **not** `pip install torch`. The runtime already has CUDA PyTorch.
3. Put session `.xlsx` / `.csv` in `data/raw/` (~104 MB, not in git).
4. `loso.json` is written after each fold.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

import torch

print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Kaggle: Settings > Accelerator > GPU. Colab: Runtime > Change runtime type > T4 GPU.")

In [ ]:
# Repo root: this notebook lives in notebooks/ when run from a clone.
root = Path.cwd()
if (root / "notebooks").is_dir() and (root / "train.py").is_file():
    pass
elif (root.parent / "train.py").is_file():
    os.chdir(root.parent)
elif (root / "train.py").is_file():
    pass
else:
    raise SystemExit("Run from the repo (folder that contains train.py), or cd there first.")
print("cwd", Path.cwd())

In [ ]:
# Install extras only. Skip torch.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26", "pandas>=2.2", "openpyxl>=3.1", "scipy>=1.11", "PyYAML>=6.0",
    "onnx>=1.16", "onnxruntime>=1.18",
])
print("ok")

## Data

Copy files into `data/raw/`.

- Kaggle: add the session dataset, then set `SRC` to that input path.
- Colab: upload the xlsx files, or copy from Drive.

In [ ]:
raw = Path("data/raw")
raw.mkdir(parents=True, exist_ok=True)

# Optional: copy from a Kaggle input or Drive folder.
SRC = os.environ.get("AURA_DATA", "")  # e.g. /kaggle/input/your-dataset
if SRC:
    src = Path(SRC)
    n = 0
    for p in list(src.glob("*.xlsx")) + list(src.glob("*.csv")):
        dest = raw / p.name
        if not dest.exists():
            shutil.copy2(p, dest)
        n += 1
    print("copied", n, "from", src)

files = sorted(p.name for p in raw.iterdir() if p.suffix.lower() in {".xlsx", ".csv"} and not p.name.startswith("~$"))
print(len(files), "session files")
print("\n".join(files))
if len(files) < 3:
    raise SystemExit("Need at least 3 session files in data/raw/.")

In [ ]:
subprocess.check_call([sys.executable, "train.py", "--loso"], cwd=str(Path.cwd()))

In [ ]:
import json

latest = Path("runs/latest.txt").read_text(encoding="utf-8").strip()
path = Path(latest) / "loso.json"
print(path)
rows = json.loads(path.read_text(encoding="utf-8"))
if isinstance(rows, dict) and "scores" in rows:
    rows = rows["scores"]
for r in rows:
    print(f"{r['session']:20s}  {100 * r['balanced_accuracy']:.1f}%")